# Regression: NHS 111 calls to GP/admission activity

This notebook starts the clinical regression workflow. The immediate aim is to test whether NHS 111 / Integrated Urgent Care activity helps explain later admission activity.

**Terminology note.** NHS England's A&E collection gives *emergency admissions*. If by "GP admissions" you mean a more specific primary-care outcome, such as GP in-hours consultations, GP out-of-hours dispositions, or referrals to primary care, use the same workflow but change the outcome column selection step.

## 0. Regression target

Baseline model:

```text
admissions_t = beta_0 + beta_1 * nhs111_calls_t + beta_2 * nhs111_calls_{t-1} + seasonal_controls + error_t
```

We will begin with monthly national totals because NHS England publishes both IUC/NHS111 and A&E emergency-admission activity as monthly statistical releases. Later we can move to system-level models if the source columns align geographically.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from wastewater.io import find_repo_root
from wastewater.clinical import (
    list_clinical_files,
    read_clinical_csv,
    quick_column_report,
    find_columns,
    aggregate_file_to_monthly,
    create_lags,
    standardise_series,
    fit_ols,
)

ROOT = find_repo_root(ROOT)
CLINICAL_RAW = ROOT / 'data' / 'clinical' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)
ROOT

## 1. Download clinical data

Run this once from the repository root before continuing:

```bash
python scripts/download_clinical_data.py
```

The script reads `clinical_sources.csv`, scrapes the NHS England statistics pages, and downloads relevant CSVs into `data/clinical/raw/`.

In [ ]:
clinical_sources = pd.read_csv(ROOT / 'clinical_sources.csv')
display(clinical_sources)

files = list_clinical_files(ROOT)
display(files.sort_values(['domain', 'year', 'path']))

## 2. Inspect candidate columns

The NHS England CSV schemas differ by collection and year. First inspect candidate call/admission columns rather than hard-coding assumptions.

In [ ]:
candidate_reports = []
for rel_path in files['path'].tolist() if not files.empty else []:
    path = ROOT / rel_path
    report = quick_column_report(path)
    candidate_reports.append({
        'path': rel_path,
        'n_cols': report['n_cols'],
        'call_like_columns': report['call_like_columns'],
        'admission_like_columns': report['admission_like_columns'],
        'gp_like_columns': report['gp_like_columns'],
    })

pd.DataFrame(candidate_reports)

## 3. Choose predictor and outcome columns

After inspecting the table above, set these two variables.

For example, possible NHS111 predictors may include calls offered, calls answered, calls triaged, or dispositions to specific services. Possible outcomes may include emergency admissions from the A&E collection, or a GP-related variable from IUC if that is the intended target.

In [ ]:
# Set these after inspecting candidate columns.
# Examples only; actual names depend on the downloaded CSV schema.
IUC_VALUE_COLUMN = None      # e.g. 'calls_offered' or another NHS111 call-volume column
AE_VALUE_COLUMN = None       # e.g. 'emergency_admissions' or a GP-specific outcome column

IUC_GROUP_COLS = []          # e.g. ['iuc_area'] if modelling by region/system
AE_GROUP_COLS = []           # e.g. ['system'] if modelling by region/system

if IUC_VALUE_COLUMN is None or AE_VALUE_COLUMN is None:
    print('Choose IUC_VALUE_COLUMN and AE_VALUE_COLUMN before running model cells.')

## 4. Build monthly analysis series

This aggregates all downloaded IUC/NHS111 files and A&E/admission files to monthly totals. Start nationally, then move to matched geographies once columns are confirmed.

In [ ]:
def build_monthly_series(domain: str, value_column: str, group_cols=None):
    group_cols = group_cols or []
    domain_files = list_clinical_files(ROOT, domain=domain)
    parts = []
    for rel_path in domain_files['path'].tolist() if not domain_files.empty else []:
        path = ROOT / rel_path
        try:
            part = aggregate_file_to_monthly(path, value_column=value_column, group_cols=group_cols)
            part['source_file'] = rel_path
            parts.append(part)
        except Exception as exc:
            print(f'Skipping {rel_path}: {exc!r}')
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

if IUC_VALUE_COLUMN and AE_VALUE_COLUMN:
    nhs111_monthly = build_monthly_series('nhs111_iuc', IUC_VALUE_COLUMN, IUC_GROUP_COLS)
    admissions_monthly = build_monthly_series('ae_emergency_admissions', AE_VALUE_COLUMN, AE_GROUP_COLS)

    display(nhs111_monthly.head())
    display(admissions_monthly.head())
else:
    nhs111_monthly = pd.DataFrame()
    admissions_monthly = pd.DataFrame()

## 5. Merge predictor and outcome

For the first pass we aggregate to national monthly totals. If region/system columns can be matched, replace this with a geography-level merge.

In [ ]:
if not nhs111_monthly.empty and not admissions_monthly.empty:
    x = nhs111_monthly.groupby('month', dropna=False)['value'].sum(min_count=1).reset_index(name='nhs111_calls')
    y = admissions_monthly.groupby('month', dropna=False)['value'].sum(min_count=1).reset_index(name='admissions')
    model_df = pd.merge(x, y, on='month', how='inner').sort_values('month')
    model_df = model_df.dropna(subset=['month', 'nhs111_calls', 'admissions'])
    model_df = standardise_series(model_df, ['nhs111_calls', 'admissions'])
    display(model_df)
else:
    model_df = pd.DataFrame()
    print('No model dataframe yet. Download data and choose columns first.')

## 6. Exploratory plots

In [ ]:
if not model_df.empty:
    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax1.plot(model_df['month'], model_df['z_nhs111_calls'], label='NHS111 calls, z-scored')
    ax1.plot(model_df['month'], model_df['z_admissions'], label='Admissions, z-scored')
    ax1.set_title('NHS111 calls and admissions, monthly national series')
    ax1.set_xlabel('Month')
    ax1.set_ylabel('z-score')
    ax1.legend()
    fig.autofmt_xdate()
    plt.show()

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(model_df['z_nhs111_calls'], model_df['z_admissions'])
    ax.set_xlabel('NHS111 calls, z-scored')
    ax.set_ylabel('Admissions, z-scored')
    ax.set_title('Contemporaneous association')
    plt.show()

## 7. Lagged regression

We test whether NHS111 calls predict admissions in the same month and at one- to three-month lags. With weekly data, this should be changed to weekly lags.

In [ ]:
if not model_df.empty:
    reg_df = create_lags(model_df, value_cols=['z_nhs111_calls'], lags=[0, 1, 2, 3], time_col='month')
    reg_df['z_nhs111_calls_lag0'] = reg_df['z_nhs111_calls']
    predictors = ['z_nhs111_calls_lag0', 'z_nhs111_calls_lag1', 'z_nhs111_calls_lag2', 'z_nhs111_calls_lag3']
    result = fit_ols(reg_df, outcome='z_admissions', predictors=predictors, add_month_dummies=True, time_col='month')
    print(result.summary())
else:
    reg_df = pd.DataFrame()

## 8. Model diagnostics and saved outputs

In [ ]:
if not reg_df.empty:
    fitted = reg_df.dropna(subset=['z_admissions', 'z_nhs111_calls_lag0', 'z_nhs111_calls_lag1', 'z_nhs111_calls_lag2', 'z_nhs111_calls_lag3']).copy()
    fitted['prediction'] = result.predict()
    fitted['residual'] = fitted['z_admissions'] - fitted['prediction']

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(fitted['month'], fitted['z_admissions'], label='Observed admissions, z')
    ax.plot(fitted['month'], fitted['prediction'], label='Predicted admissions, z')
    ax.set_title('Observed vs fitted admissions')
    ax.legend()
    fig.autofmt_xdate()
    plt.show()

    out_path = PROCESSED / 'nhs111_admissions_regression_frame.csv'
    reg_df.to_csv(out_path, index=False)
    print(f'Saved regression frame to {out_path.relative_to(ROOT)}')

## 9. Next refinements

1. Confirm whether the outcome should be emergency admissions, GP out-of-hours dispositions, GP consultations, or another primary-care measure.
2. Move from monthly to weekly data if source files support it.
3. Fit geography-level models if IUC areas can be mapped to A&E systems.
4. Add wastewater pathogen signals as additional predictors once the clinical baseline is stable.
5. Compare OLS, Poisson/negative binomial, and autoregressive models.